# 📞 Telco Customer 360 — Churn, Voix du Client & Rétention
## Plateforme client intelligente sur Azure : prédiction de churn, analyse d'avis réels (NLP + RAG) et moteur de rétention

---

### 🎯 Contexte

Vous êtes **Data / IA Engineer** dans l'équipe Expérience Client d'un opérateur télécom. Chaque mois, des milliers de clients résilient (**churn**). Retenir un client existant coûte 5 à 7 fois moins cher que d'en acquérir un nouveau : la rétention est un enjeu business majeur.

On vous confie une plateforme **Customer 360** qui doit répondre à trois questions :
1. **Qui va partir ?** → un modèle prédictif de churn
2. **Pourquoi les clients partent ?** → l'explicabilité du modèle (SHAP) **+ l'analyse de vrais avis clients** collectés sur le web
3. **Que proposer pour retenir ?** → un moteur de rétention (fiches conseiller) + un assistant qui interroge les avis en langage naturel (**RAG**)

### 🧩 L'idée clé : une architecture en deux couches reliées par le *motif*

Ce projet combine deux mondes de données qui **ne parlent pas des mêmes individus** — et c'est assumé :

| Couche | Données | Rôle | Répond à |
|---|---|---|---|
| **1. Client (structuré)** | Dataset Telco Churn (Kaggle) | Modèle prédictif + SHAP | *Qui* va partir, *pourquoi* (au niveau individuel) |
| **2. Voix du client (non structuré, RÉEL)** | Vrais avis clients (Google Play, Reddit) | NLP + sentiment + RAG | *Ce que* les clients reprochent (au niveau marché/opérateur) |

⚠️ **Le pont entre les deux, ce n'est PAS le client, c'est le motif** (réseau, facturation, résiliation, service client). Les vrais avis portent sur des opérateurs, pas sur les clients anonymes du dataset Kaggle : on ne peut donc pas les joindre par `customerID`. On relie donc les deux couches par thème : SHAP dit « ce client risque de partir pour un profil *facturation* », et le corpus d'avis réels dit « voilà concrètement ce que les gens reprochent sur la *facturation* ». **Savoir expliquer et défendre ce choix est un excellent point d'entretien** (maturité sur les limites des données).

### 💡 Différence avec le Telco « d'origine »

Dans la version de départ, les tickets de réclamation étaient **fabriqués** (données synthétiques collées aux clients Kaggle). Ici on les remplace par de **vrais avis** collectés via API/scraping, et on ajoute deux briques absentes de l'original : la vraie chaîne **data engineering** (ingestion robuste, modélisation en couches, tests, orchestration, CI/CD) et l'**IA générative** (embeddings + RAG sourcé). On passe d'un projet ML à une plateforme end-to-end.

### 🧰 Stack technique

**Colonne vertébrale Azure** (application concrète de l'AZ-900) : Blob Storage, Azure Functions, Azure AI Language ou Azure OpenAI, App Service, Microsoft Entra ID, Cost Management.

**Briques data/IA** : Python, pandas, scikit-learn, XGBoost, SHAP ; `google-play-scraper` + `praw` (Reddit) pour l'ingestion ; dbt (ou SQL structuré) pour la transformation ; un modèle d'embeddings + un vector store (`chromadb`/`faiss`) pour le RAG ; Streamlit pour l'app.

> 🔁 **Alternative 100% locale et gratuite** (si vous ne voulez pas de crédit Azure) : DuckDB au lieu de Blob, un LLM via API gratuite (Groq/Mistral) ou Ollama au lieu d'Azure OpenAI, Prefect au lieu d'Azure Functions, Streamlit Community Cloud au lieu d'App Service. L'énoncé signale ces équivalences à chaque étape concernée.

### 📦 Livrables attendus

| # | Livrable | Format |
|---|---|---|
| 1 | Repo Git structuré (pipeline + app) + schéma d'architecture | GitHub + image |
| 2 | Modèle de churn entraîné + rapport de performance | `.joblib` + métriques |
| 3 | Base d'avis réels enrichie (motif + sentiment + entités) | table / parquet |
| 4 | Index vectoriel + moteur RAG sourcé + rapport d'évaluation | vector store + notebook |
| 5 | « Liste rouge » des clients prioritaires | `.csv` |
| 6 | Fiches de rétention générées automatiquement | texte / JSON |
| 7 | Dashboard interactif déployé (URL publique) | App Service / Streamlit Cloud |
| 8 | README + bilan des coûts + pitch | markdown |

### 🗺️ Vue d'ensemble du flux

```
COUCHE CLIENT      Kaggle Telco → Blob/DuckDB → EDA → XGBoost → SHAP ─┐
                                                                        ├→ Liste rouge → Fiches conseiller → Dashboard
COUCHE VOIX CLIENT  Google Play + Reddit → stockage → dbt → NLP/LLM ────┘        ↑ (RAG sourcé)
                                                          └→ embeddings → vector store → RAG ───┘
                    orchestration (Azure Function / Prefect) + CI/CD en transverse
```

⏱️ **Planning indicatif (6-7 semaines à temps partiel)** : S1 Étapes 1-3 · S2 Étapes 4-5 · S3 Étapes 6-7 · S4 Étapes 8-10 · S5 Étapes 11-13 · S6-7 Étapes 14-16.

---
# Étape 1 — Mise en place de l'environnement (Azure + repo)

**🎓 Objectif** : partir sur des bases pro — un cloud gouverné côté coûts, un projet versionné côté code. C'est le domaine 1 & 3 de l'AZ-900 mis en pratique.

**Ce que vous devez faire :**
1. Créer un compte **Azure for Students** (100$ de crédit, sans carte, avec l'email universitaire) ou un compte d'essai.
2. Créer un **groupe de ressources** `rg-telco360` en région `France Central`. Comprendre pourquoi on regroupe : un groupe de ressources se facture, se gouverne et se supprime d'un bloc.
3. Configurer un **budget** avec alertes à 50% et 80% dans **Cost Management** — le réflexe qui évite la VM oubliée à 300€.
4. Créer le **repo Git** avec une structure claire dès le départ : `ingestion/`, `transform/`, `modeling/`, `enrichment/`, `rag/`, `app/`, `tests/`, `flows/`, `docs/`.
5. Mettre en place la **gestion des secrets** : un `.env` (dans le `.gitignore`, jamais commité) + un `.env.example` documenté. Aucune clé (Azure, Reddit, LLM) ne doit apparaître dans le code.

**🔍 Points d'attention :**
- Pourquoi `France Central` plutôt que `East US` ? (latence, souveraineté des données, RGPD)
- Différence entre abonnement, groupe de gestion et groupe de ressources ?

**🔁 Variante locale** : pas d'Azure → vous créez juste le repo et un dossier `data/`. La gouvernance des coûts devient sans objet, mais gardez la discipline secrets.

✅ **Critère de réussite** : groupe de ressources + budget en place (capture d'écran), repo initialisé avec la structure et les secrets hors du code.

*Valorise : Data Engineer (gouvernance, structure projet).*

---
# Étape 2 — Ingestion des données clients (Kaggle → stockage)

**🎓 Objectif** : comprendre le stockage objet, les tiers d'accès et la redondance (domaine 2 de l'AZ-900), et manipuler un SDK cloud en Python.

**Ce que vous devez faire :**
1. Récupérer le dataset **Telco Customer Churn** de Kaggle (~7 000 clients, 21 variables : profil, services, facturation, et la cible `Churn`). **C'est la base que vous gardez telle quelle** — elle alimente la couche churn.
2. Créer un **compte de stockage** Azure avec redondance **LRS**. Se demander : pourquoi LRS suffit ici, et dans quel cas exiger du GRS (géo-redondance) ?
3. Créer deux conteneurs : `raw-data` (brut) et `processed-data` (nettoyé).
4. Téléverser le CSV dans `raw-data` — d'abord via le portail, puis **en Python** avec `azure-storage-blob` (c'est cette version que le recruteur veut voir).
5. Configurer une règle de **lifecycle management** : le brut passe de *Hot* à *Cool* après 30 jours.

**🔒 Sécurité** : jamais la chaîne de connexion en dur → variable d'environnement, et en production **Azure Key Vault**.

**🔁 Variante locale** : charger le CSV dans une base **DuckDB** (`raw_churn`). Comprendre pourquoi DuckDB est idéal ici (colonne, SQL complet, zéro serveur, lit le parquet).

✅ **Critère de réussite** : lire le CSV clients directement depuis le stockage (Blob ou DuckDB) dans votre notebook.

*Valorise : Data Engineer (stockage, SDK cloud).*

---
# Étape 3 — Exploration et préparation des données clients (EDA)

**🎓 Objectif** : comprendre les données avant de modéliser — l'étape que les débutants bâclent et que les seniors soignent.

**Ce que vous devez faire :**
1. **Audit qualité** : le piège classique du dataset est `TotalCharges`, stockée en texte avec des valeurs vides pour les nouveaux clients. L'identifier, le corriger, et expliquer pourquoi ces valeurs sont vides (regarder `tenure`).
2. **Analyse de la cible** : taux de churn global ? dataset déséquilibré ? conséquences pour la modélisation ?
3. **Analyses univariée et bivariée** — au minimum : ancienneté (`tenure`) selon le churn, facture mensuelle selon le churn, taux de churn par type de contrat, par mode de paiement, par type d'internet.
4. **Synthèse métier** : 4-5 phrases décrivant le « portrait-robot » du client qui résilie. C'est cette traduction chiffres → business qui fait la différence en entretien.

**Résultat attendu** (à retrouver vous-même) : les clients en contrat mensuel, récents, en fibre et à facture élevée churnent massivement ; les clients engagés 2 ans presque jamais.

✅ **Critère de réussite** : 3+ visualisations propres et titrées + synthèse métier écrite.

*Valorise : Data Scientist (EDA, lecture métier).*

---
# Étape 4 — Modèle de churn (XGBoost) et explicabilité (SHAP)

**🎓 Objectif** : un modèle performant **ET** explicable — un score sans explication est inutilisable par le métier.

**Ce que vous devez faire :**
1. **Feature engineering** — au moins 3 variables métier : `nb_services` (nombre de services souscrits), `charge_par_service` (facture / nb services, détecte le sentiment de « payer trop cher »), `nouveau_client` (ancienneté ≤ 6 mois).
2. **Encodage** des catégorielles (justifier LabelEncoder vs One-Hot).
3. **Split** train/test **stratifié** 80/20 (pourquoi stratifié ? à cause du déséquilibre).
4. **Entraîner un XGBoost**, gérer le déséquilibre avec `scale_pos_weight` (savoir expliquer ce que fait ce paramètre).
5. **Évaluer** : AUC, precision, recall, matrice de confusion. Trancher **du point de vue métier** : precision ou recall ? (rater un client qui part coûte plus cher qu'appeler un client qui serait resté → on privilégie le recall).
6. **SHAP** : un `summary_plot` global (quelles variables pilotent le churn ?) + une fonction `expliquer_client(id)` qui retourne, pour un client, sa probabilité de churn et ses 3 principaux facteurs de risque.
7. Sauvegarder le modèle (`.joblib`).

✅ **Critère de réussite** : AUC ≥ 0.83 + fonction d'explication individuelle fonctionnelle + modèle sauvegardé.

*Valorise : Data Scientist / ML Engineer (modélisation, explicabilité).*

---
# Étape 5 — Ingestion des VRAIS avis clients (le cœur data engineer)

**🎓 Objectif** : remplacer les tickets synthétiques de la version d'origine par de vraies données, et maîtriser l'appel d'API, la gestion d'erreurs, le dédoublonnage — le quotidien d'un data engineer.

**Ce que vous devez faire :**
1. **Source n°1 — Google Play (principale)** : récupérer les avis des applis des opérateurs français (Orange, SFR, Free, Bouygues) avec la lib `google-play-scraper`. Volumineux, en français, avec **note + texte + date**.
2. **Source n°2 — Reddit (complément)** : via l'API officielle et la lib `praw` (créer une app Reddit de type script → `client_id` + `client_secret`). Rechercher les mentions des opérateurs sur r/france et subs dédiés. Moins dense mais montre la consommation d'une **API authentifiée (OAuth)**.
3. **Ingestion robuste** : timeout, **retries avec backoff**, gestion du rate limit, logging.
4. **Dédoublonnage / idempotence** : clé d'unicité par avis (id source ou hash). Relancer le script ne doit **pas** créer de doublons.
5. **Normalisation** : fusionner les deux sources hétérogènes dans un schéma commun — `id`, `source`, `operateur`, `note` (si dispo), `texte`, `date`, `ingested_at`. Conserver aussi le brut.
6. Stocker le résultat dans `raw-data` (ou DuckDB en local).

**🔍 Points d'attention :**
- Fusionner deux sources de formats différents (avis notés vs discussions libres) est un **vrai travail de data engineer** — mettez-le en avant.
- **Éthique/légal** : privilégier les API officielles (Reddit) et les libs dédiées (Google Play) ; éviter le scraping de sites qui l'interdisent (ex. Trustpilot). Documenter ce choix.

✅ **Critère de réussite** : lancer l'ingestion deux fois → aucun doublon ; un DataFrame d'avis réels normalisés des 4 opérateurs.

*Valorise : Data Engineer (ingestion multi-sources, robustesse, idempotence).*

---
# Étape 6 — Modélisation en couches et qualité des données (dbt)

**🎓 Objectif** : découvrir **dbt** et les **tests de données** — la compétence data eng la plus valorisée et la plus rare en portfolio.

**Ce que vous devez faire :**
1. Installer **dbt-duckdb** (ou dbt sur la base de votre choix) et l'initialiser.
2. Construire les couches sur les avis :
   - `stg_avis` (staging) : nettoyage, typage des dates, dédoublonnage SQL, normalisation des opérateurs et des sources.
   - `mart_avis` (mart) : table propre prête pour l'enrichissement et l'app.
3. Ajouter des **tests dbt** : `unique` + `not_null` sur l'id, `not_null` sur `texte`/`operateur`, test d'acceptation sur la note (dans l'intervalle attendu) et sur la date (pas dans le futur).
4. Générer la **doc dbt** (`dbt docs generate`) — le lineage visuel fait son effet en entretien.

**🔍 Point d'attention** : pourquoi transformer en SQL/dbt plutôt qu'en pandas ? (lisibilité, testabilité, lineage, exécution dans l'entrepôt)

**💡 Version légère** : si dbt vous ralentit, faites les couches en SQL/Python propre avec des **assertions de qualité** explicites (mais dbt est un vrai plus CV).

✅ **Critère de réussite** : `dbt run` + `dbt test` au vert, lineage généré.

*Valorise : Data Engineer (transformation, tests, lineage).*

---
# Étape 7 — Enrichissement NLP/LLM des avis (motif + sentiment + entités)

**🎓 Objectif** : transformer du texte brut en signal exploitable, et comparer une approche artisanale à un service managé / LLM.

**Ce que vous devez faire :**
1. **Baseline locale** : un classifieur de motifs simple (TF-IDF + régression logistique, ou mots-clés) sur les 4 motifs (`réseau`, `facturation`, `résiliation`, `service_client`). Étiquetez un petit échantillon à la main pour mesurer l'accuracy.
2. **Version LLM (Azure OpenAI, ou API gratuite)** : pour chaque avis, produire une sortie **structurée (JSON)** : `motif`, `sentiment` (négatif / très négatif / neutre / positif), `entites` (offres, technos, mots-clés cités). **Forcer le format** (JSON mode / schéma Pydantic) et gérer les sorties non parsables (retry/fallback).
3. **Idempotence + coût** : ne jamais ré-enrichir un avis déjà traité ; logger les tokens / le coût par run ; traiter par lots.
4. **Comparaison** : baseline vs LLM — quand préférer un service managé/LLM vs son propre modèle ? (coût, maintenance, RGPD, personnalisation — question type AZ-900 SaaS vs build).
5. Stocker les résultats dans `article_enrichment` joint sur l'id d'avis.

**🔍 Point d'attention** : gérer les hallucinations sur le motif/les entités → une **liste fermée** de motifs dans le prompt limite le problème.

✅ **Critère de réussite** : chaque avis enrichi d'un JSON valide (`motif`, `sentiment`, `entites`) + un paragraphe comparant baseline et LLM.

*Valorise : IA Engineer + Data Scientist (NLP, LLM structuré).*

---
# Étape 8 — Indexation vectorielle (préparation du RAG)

**🎓 Objectif** : comprendre embeddings, chunking et vector store — les fondations d'un RAG.

**Ce que vous devez faire :**
1. **Chunking** : découper le texte des avis en morceaux cohérents (ici un avis est souvent court → souvent 1 avis = 1 chunk, mais gérez les longs). Comprendre le compromis taille/overlap.
2. **Embeddings** : calculer le vecteur de chaque chunk (modèle d'embeddings via API, ou `sentence-transformers` en local et gratuit — multilingue pour le français).
3. **Vector store** (`chromadb` ou `faiss`) : stocker les vecteurs **avec leurs métadonnées** (id avis, opérateur, motif, sentiment, date, source) — indispensable pour **citer la source** et **filtrer** ensuite.
4. **Idempotence** : ne ré-indexer que les nouveaux chunks.

**🔍 Point d'attention** : garder l'opérateur et le motif en métadonnée permet un RAG **filtré** (« uniquement les avis SFR sur la facturation »), bien plus utile qu'une recherche brute.

✅ **Critère de réussite** : une recherche sémantique renvoie des avis pertinents avec leur opérateur/motif/source.

*Valorise : IA Engineer (embeddings, vector store).*

---
# Étape 9 — Le moteur RAG (retrieval + génération + garde-fous)

**🎓 Objectif** : assembler un RAG complet et **honnête** — qui cite ses sources et refuse d'inventer.

**Ce que vous devez faire :**
1. Fonction `repondre(question, operateur=None, motif=None)` : embed la question → **retrieve** les k avis les plus proches (avec filtre optionnel opérateur/motif) → injecte dans le prompt → **génère** la réponse.
2. **Citations obligatoires** : la réponse renvoie les avis sources utilisés (extrait + opérateur + date).
3. **Garde-fous** : si le contexte récupéré ne contient pas la réponse, le modèle doit le dire (« pas d'information là-dessus dans les avis collectés ») plutôt que d'halluciner. C'est LE point qui sépare une démo d'un produit.
4. Soigner le **prompt système** : rôle (« assistant d'analyse de la voix du client télécom »), obligation de s'appuyer uniquement sur le contexte, format de sortie.

**Exemples de questions cibles** : « Que reprochent le plus les clients à SFR sur la facturation ? », « Quels sont les griefs récurrents sur le réseau chez Free ? ».

✅ **Critère de réussite** : réponses sourcées et pertinentes ; une question hors-corpus → refus propre, pas d'invention.

*Valorise : IA Engineer (RAG, garde-fous, prompt engineering).*

---
# Étape 10 — Évaluation du RAG

**🎓 Objectif** : mesurer la qualité d'un système IA au lieu de « ça a l'air de marcher » — très rare en portfolio, très valorisé.

**Ce que vous devez faire :**
1. Construire un petit **jeu d'évaluation** : 15-25 questions dont vous connaissez la réponse (+ quelques questions pièges hors-corpus).
2. Évaluer deux dimensions :
   - **Retrieval** : les bons avis sont-ils dans le top-k ? (hit rate)
   - **Génération** : la réponse est-elle fidèle au contexte (pas d'hallucination) et pertinente ? Utiliser un **LLM-as-judge** ou une grille manuelle.
3. Faire varier **un** paramètre (taille de chunk, ou k) et montrer l'impact chiffré — c'est ça l'approche ingénieur.
4. Rédiger un court rapport (métriques + limites).

✅ **Critère de réussite** : un tableau de métriques + au moins une comparaison de configuration documentée.

*Valorise : IA Engineer (évaluation, rigueur).*

---
# Étape 11 — La « liste rouge » : croisement churn × motifs

**🎓 Objectif** : c'est ici que le projet devient un produit. On croise les deux couches pour prioriser l'action — **par le motif, pas par l'individu**.

**Ce que vous devez faire :**
1. Scorer **toute la base clients** avec le modèle de churn (étape 4).
2. Pour chaque client à risque, déterminer son **motif dominant** à partir de ses facteurs SHAP (ex. profil « facture élevée / charge par service élevée » → motif *facturation*). Définissez une règle de correspondance SHAP → motif.
3. **Enrichir le motif avec la voix du client** : pour le motif dominant, récupérer via le RAG/agrégation les **griefs réels** correspondants (verbatims types, sentiment moyen sur ce motif).
4. Construire la **liste rouge** : clients `proba_churn ≥ 0.6`, triés par risque, enrichis (ancienneté, contrat, facture, motif dominant, top 3 SHAP, griefs réels associés).
5. Exporter en CSV dans `processed-data`.

**🔍 Point d'attention métier** : pourquoi 0.6 et pas 0.5 ? Raisonner en **capacité de traitement** : si l'équipe rétention n'appelle que 50 clients/semaine, le seuil s'ajuste à cette contrainte, pas à une convention statistique.

✅ **Critère de réussite** : un CSV exploitable tel quel par une équipe métier, où chaque client à risque est relié à des griefs réels via son motif.

*Valorise : Data Scientist (industrialisation du score, sens produit).*

---
# Étape 12 — Moteur de rétention : fiches conseiller générées par LLM

**🎓 Objectif** : combiner ML prédictif et IA générative dans un cas d'usage concret et **cadré**.

**Ce que vous devez faire :**
1. Définir une **matrice d'actions de rétention** : pour chaque motif, une action type (facturation → audit de facture + remboursement d'options non souscrites ; réseau → diagnostic prioritaire + geste commercial ; etc.).
2. Créer `fiche_conseiller(client)` qui produit une fiche structurée : identité, risque de churn, top facteurs SHAP, **motif dominant + verbatims réels** issus du corpus, action recommandée.
3. **Version LLM (Azure OpenAI / API)** : générer une fiche **rédigée** en langage naturel. Prompt soigné : rôle (« assistant pour conseillers clientèle télécom »), contexte injecté (données client + griefs réels), format imposé.
4. **Garde-fous** : contraindre la sortie à une **liste d'actions autorisées** dans le prompt pour éviter que le LLM invente une remise inexistante. Documenter cette réflexion (problématique réelle des LLM en entreprise).

✅ **Critère de réussite** : 10 fiches générées pour le top 10 de la liste rouge, appuyées sur de vrais verbatims, sans hallucination d'offre.

*Valorise : IA Engineer (LLM appliqué, cadrage, garde-fous).*

---
# Étape 13 — Orchestration et automatisation

**🎓 Objectif** : passer d'une suite de scripts à un **pipeline orchestré**, planifié, avec retries et observabilité — le cœur du métier data engineer.

**Ce que vous devez faire :**
1. Assembler les étapes en un **pipeline** : ingestion avis → transformation (dbt) → enrichissement LLM → indexation → (re)scoring churn.
2. **Version Azure** : une **Azure Function** avec **Blob Trigger** qui relance le traitement dès qu'un nouveau fichier arrive dans `raw-data` (concept serverless de l'AZ-900).
3. **Version locale/portable** : un **flow Prefect** avec tasks, **retries** et **timeouts** sur les étapes fragiles (réseau, LLM), et une **planification** (ex. quotidienne).
4. Exploiter l'observabilité : voir les runs, les échecs, les durées (dashboard Prefect ou logs/Monitor Azure). Montrer plusieurs runs réussis.

**🔍 Point d'attention** : différence entre une orchestration (Prefect/Function) et un simple `cron` ? (dépendances, retries, visibilité, reprise sur échec)

✅ **Critère de réussite** : un pipeline qui tourne de bout en bout, automatisé, avec retries et suivi des exécutions.

*Valorise : Data Engineer (orchestration, serverless).*

---
# Étape 14 — Dashboard et déploiement

**🎓 Objectif** : passer du notebook au produit déployé — la compétence qui distingue un candidat. Une URL montrable depuis le téléphone en entretien.

**Ce que vous devez faire :**
1. Une app **Streamlit** avec 4 vues :
   - **Portefeuille** : KPIs globaux (taux de churn prédit, nb de clients en liste rouge, répartition des motifs, sentiment moyen par opérateur).
   - **Client** : recherche par `customerID` → score, facteurs SHAP, motif dominant, fiche de rétention.
   - **Voix du client** : dashboard des avis réels (volumes par opérateur/motif, sentiment, top griefs).
   - **Assistant RAG** : poser une question → réponse sourcée (avis cités).
2. L'app lit ses données **depuis le stockage** (Blob ou base), pas des CSV locaux — c'est ce qui en fait une vraie app cloud.
3. **Déploiement** : **Azure App Service** (plan F1/B1) via Git ou GitHub Actions ; **ou** Streamlit Community Cloud en version locale. Secrets via la config sécurisée (jamais en dur).
4. **Sécuriser** (Azure) : activer l'authentification **Microsoft Entra ID** (« Easy Auth ») pour restreindre l'accès.

✅ **Critère de réussite** : une URL publique fonctionnelle (protégée par login si Azure), les 4 vues opérationnelles.

*Valorise : Data/IA Engineer (déploiement, produit).*

---
# Étape 15 — Industrialisation : tests et CI/CD

**🎓 Objectif** : la couche qui transforme un « projet perso » en « projet pro ».

**Ce que vous devez faire :**
1. **Tests** `pytest` sur les fonctions clés : normalisation des avis, dédoublonnage, parsing du JSON LLM, retrieval du RAG, règle SHAP → motif.
2. **GitHub Actions** : un workflow qui, à chaque push, lance le lint (`ruff`) + les tests. Le badge vert sur le repo est un signal fort.
3. **Qualité des données en CI** : optionnellement, faire tourner `dbt test` dans le pipeline.

✅ **Critère de réussite** : tests qui passent en local et en CI, workflow GitHub Actions vert.

*Valorise : Data/ML Engineer (tests, CI/CD, fiabilité).*

---
# Étape 16 — Documentation, coûts et pitch

**🎓 Objectif** : un projet non documenté n'existe pas. Savoir parler coûts et architecture = crédibilité immédiate.

**Ce que vous devez faire :**
1. **Schéma d'architecture** propre (draw.io / excalidraw) : les deux couches, le flux complet (ingestion → stockage → dbt → modèle/enrichissement → vector store → RAG → app), avec orchestration et CI/CD en transverse.
2. **Bilan des coûts** (si Azure) : relever dans Cost Management la consommation service par service ; estimer le coût mensuel à l'échelle (ex. 100 000 clients).
3. **README** soigné : contexte, architecture deux couches (avec la limite assumée du pont par motif), résultats (AUC, métriques RAG, exemples de fiches, captures), instructions de reproduction.
4. **Pitch de 2 minutes** (écrivez-le) : *« J'ai construit une plateforme Customer 360 qui prédit le churn (XGBoost + SHAP), analyse de vrais avis clients collectés via API (pipeline data eng testé et orchestré, enrichi par LLM), et permet d'interroger ces avis via un RAG sourcé évalué, le tout déployé avec authentification. »*

### 🧭 Ce que chaque poste retiendra

| Poste visé | À mettre en avant |
|---|---|
| **Data Engineer** | Ingestion multi-sources robuste & idempotente, dbt + tests, orchestration, CI/CD |
| **IA Engineer** | Enrichissement LLM structuré, embeddings, RAG sourcé + garde-fous, évaluation |
| **Data Scientist / ML** | Churn XGBoost, SHAP, seuils métier, industrialisation du score |

### 🧭 Correspondance AZ-900 (si version Azure)

| Domaine AZ-900 | Où vous l'avez pratiqué |
|---|---|
| Concepts cloud (IaaS/PaaS/SaaS, responsabilité partagée) | Function (serverless), App Service (PaaS), AI Language/OpenAI (SaaS) |
| Architecture (régions, groupes de ressources, abonnements) | Étape 1 |
| Stockage (tiers, redondance, lifecycle) | Étape 2 |
| Identité et sécurité (Entra ID, Key Vault) | Étapes 2 et 14 |
| Gouvernance (Cost Management, budgets, Monitor) | Étapes 1 et 16 |

Bon courage 🚀